In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn

from pathlib import Path
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display

RANDOM_STATE = 42
OUT = Path('/content/project1_results') if Path('/content').exists() else Path('project1_results')
OUT.mkdir(parents=True, exist_ok=True)
print('scikit-learn:', sklearn.__version__)
print('Folder kết quả:', OUT)

In [ ]:
housing = fetch_california_housing(as_frame=True)
X = housing.data.copy()
y = housing.target.copy() * 100_000

df = X.copy()
df['MedianHouseValueUSD'] = y
print('Số hàng, số cột:', df.shape)
print('Tên đặc trưng:', X.columns.tolist())
display(df.head(10))
print('Số giá trị bị thiếu theo cột:')
display(df.isna().sum().to_frame('Missing values'))

In [ ]:
display(df.describe().T)
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(numeric_only=True), cmap='coolwarm', center=0, annot=True, fmt='.2f')
plt.title('Correlation matrix - California Housing')
plt.tight_layout()
plt.savefig(OUT / 'correlation_heatmap.png', dpi=160)
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print('Train:', X_train.shape, '| Test:', X_test.shape)
print('Dữ liệu mục tiêu train:', y_train.shape, '| test:', y_test.shape)

In [ ]:
linear_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])
linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)
print('Đã train xong Linear Regression')

In [ ]:
rf_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', RandomForestRegressor(
        n_estimators=100,
        max_depth=20,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
print('Đã train xong Random Forest')

In [ ]:
def evaluate_model(name, y_true, y_pred):
    return {
        'Model': name,
        'MAE_USD': mean_absolute_error(y_true, y_pred),
        'RMSE_USD': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred)
    }

comparison = pd.DataFrame([
    evaluate_model('Linear Regression', y_test, linear_pred),
    evaluate_model('Random Forest', y_test, rf_pred)
]).sort_values('MAE_USD').reset_index(drop=True)

display(comparison.round({'MAE_USD': 2, 'RMSE_USD': 2, 'R2': 4}))
comparison.to_csv(OUT / 'model_comparison.csv', index=False)

best_name = comparison.iloc[0]['Model']
best_model = linear_model if best_name == 'Linear Regression' else rf_model
best_pred = linear_pred if best_name == 'Linear Regression' else rf_pred
print('Mô hình có MAE thấp hơn:', best_name)

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, best_pred, s=9, alpha=0.25)
low = min(float(np.min(y_test)), float(np.min(best_pred)))
high = max(float(np.max(y_test)), float(np.max(best_pred)))
plt.plot([low, high], [low, high], 'r--', linewidth=1.5, label='Perfect prediction')
plt.xlabel('Actual district median house value (USD)')
plt.ylabel('Predicted value (USD)')
plt.title('Actual vs predicted - ' + best_name)
plt.legend()
plt.tight_layout()
plt.savefig(OUT / 'actual_vs_predicted.png', dpi=160)
plt.show()

predictions = pd.DataFrame({
    'Actual_USD': y_test.to_numpy(),
    'LinearRegression_Pred_USD': linear_pred,
    'RandomForest_Pred_USD': rf_pred
}, index=y_test.index)
predictions.index.name = 'row_index'
predictions.to_csv(OUT / 'test_predictions.csv')
display(predictions.head(10))

In [ ]:
feature_importance = pd.Series(
    rf_model.named_steps['model'].feature_importances_,
    index=X_train.columns,
    name='Importance'
).sort_values(ascending=False)
display(feature_importance.to_frame())
feature_importance.to_csv(OUT / 'feature_importance.csv')

plt.figure(figsize=(8, 4.5))
feature_importance.sort_values().plot(kind='barh')
plt.title('Random Forest feature importance')
plt.xlabel('Relative importance')
plt.tight_layout()
plt.savefig(OUT / 'feature_importance.png', dpi=160)
plt.show()

In [ ]:
joblib.dump(best_model, OUT / 'best_house_price_model.joblib')

import shutil
zip_path = shutil.make_archive(str(OUT), 'zip', root_dir=str(OUT))
print('Đã lưu:', zip_path)
print('Các file:')
for p in sorted(OUT.iterdir()):
    print(' -', p.name)

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Không chạy trong Colab; xem file ZIP ở:', zip_path)